# --- 1. Imports and Setup ---

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
import os
import pickle
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings('ignore')
%matplotlib inline


# --- 2. Load Data ---
# UPDATE THIS PATH TO WHERE YOU UPLOADED THE CSV IN YOUR WORKSPACE

In [0]:
df = pd.read_csv("/Workspace/Users/rsangramofficial@gmail.com/EDA/Shopper Spectrum/online_retail.csv")   

display(df.head())


# --- 3. Data Preprocessing ---

In [0]:


df['CustomerID'] = df['CustomerID'].astype('float').astype('Int64')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Remove missing CustomerID
df = df.dropna(subset=['CustomerID'])

# Remove cancelled invoices
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]

# Remove negative/zero Quantity and UnitPrice
df = df[df['Quantity'] > 0]
df = df[df['UnitPrice'] > 0]

# Create TotalPrice
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

print("Cleaned data shape:", df.shape)

# --- 4. Exploratory Data Analysis (EDA) ---
# Transactions by Country (Top 15)

In [0]:

country_counts = df['Country'].value_counts().head(15)
plt.figure(figsize=(12,6))
sns.barplot(x=country_counts.values, y=country_counts.index)
plt.title('Top 15 Countries by Transactions')
display(plt)

# Top 10 selling products

In [0]:

top_products = df.groupby('Description')['Quantity'].sum().sort_values(ascending=False).head(10)
plt.figure(figsize=(10,6))
sns.barplot(x=top_products.values, y=top_products.index)
plt.title('Top 10 Products by Quantity Sold')
display(plt)

# Monthly revenue trend

In [0]:


monthly_sales = df.set_index('InvoiceDate').resample('M')['TotalPrice'].sum()
plt.figure(figsize=(12,5))
monthly_sales.plot()
plt.title('Monthly Revenue Trend')
plt.ylabel('Revenue')
display(plt)

# Monetary distribution per customer

In [0]:

customer_monetary = df.groupby('CustomerID')['TotalPrice'].sum()
plt.figure(figsize=(10,5))
sns.histplot(customer_monetary, bins=50, kde=True)
plt.title('Distribution of Total Spend per Customer')
display(plt)

# --- 5. RFM Calculation ---

In [0]:


latest_date = df['InvoiceDate'].max() + timedelta(days=1)

rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (latest_date - x.max()).days,  # Recency
    'InvoiceNo': 'nunique',                                # Frequency
    'TotalPrice': 'sum'                                    # Monetary
})

rfm.columns = ['Recency', 'Frequency', 'Monetary']
rfm = rfm[rfm['Monetary'] > 0]

display(rfm.describe())

# --- 6. Standardization & Clustering ---

In [0]:

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm)

# Elbow + Silhouette
inertias = []
sil_scores = []
K = range(2, 11)
for k in K:
    kmeans_temp = KMeans(n_clusters=k, random_state=42)
    kmeans_temp.fit(rfm_scaled)
    inertias.append(kmeans_temp.inertia_)
    sil_scores.append(silhouette_score(rfm_scaled, kmeans_temp.labels_))

plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(K, inertias, 'bx-')
plt.title('Elbow Method')
plt.xlabel('k')
plt.subplot(1,2,2)
plt.plot(K, sil_scores, 'bx-')
plt.title('Silhouette Score')
plt.xlabel('k')
display(plt)

# Use k=4 (typical best for this dataset)
optimal_k = 4
kmeans = KMeans(n_clusters=optimal_k, random_state=42)
rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)

# --- 7. Cluster Interpretation ---

In [0]:

cluster_profile = rfm.groupby('Cluster')[['Recency','Frequency','Monetary']].mean()
display(cluster_profile)

# Manual labeling (adjust mapping after seeing cluster_profile if needed)
cluster_labels = {
    0: 'High-Value',
    1: 'At-Risk',
    2: 'Regular',
    3: 'Occasional'
}
rfm['Segment'] = rfm['Cluster'].map(cluster_labels)
display(rfm['Segment'].value_counts())

# Scatter plot
plt.figure(figsize=(10,6))
sns.scatterplot(data=rfm, x='Recency', y='Monetary', hue='Segment', palette='deep')
plt.title('Customer Segments')
display(plt)


# --- 8. Item-based Collaborative Filtering ---

In [0]:

pivot = df.pivot_table(index='Description', columns='CustomerID', values='Quantity', 
                       aggfunc='sum', fill_value=0)

item_similarity = cosine_similarity(pivot)
similarity_df = pd.DataFrame(item_similarity, index=pivot.index, columns=pivot.index)

# Test recommendation
def get_recommendations(product_name, top_n=5):
    if product_name not in similarity_df.index:
        return ["Product not found"]
    similar = similarity_df[product_name].sort_values(ascending=False)[1:top_n+1]
    return similar.index.tolist()

print(get_recommendations("WHITE HANGING HEART T-LIGHT HOLDER"))

# --- 9. Save Models to Your Workspace Folder ---

In [0]:

model_path = "/Workspace/Users/rsangramofficial@gmail.com/EDA/Shopper Spectrum/models/"

# Create directory if it doesn't exist
os.makedirs(model_path, exist_ok=True)

pickle.dump(similarity_df, open(model_path + 'similarity_df.pkl', 'wb'))
pickle.dump(kmeans, open(model_path + 'kmeans.pkl', 'wb'))
pickle.dump(scaler, open(model_path + 'scaler.pkl', 'wb'))
pickle.dump(cluster_labels, open(model_path + 'cluster_labels.pkl', 'wb'))

print(f"All models saved successfully to: {model_path}")